# CeO2 Demo: ImageD11 `.par` → pyFAI `.poni` Conversion Solutions

**Author:** DeepSeek V4 Pro (opencode), June 2026

---

### Plan

1. Define an **ImageD11 geometry** for an **Eiger4M** detector (flip `(−1,0,0,−1)`).
2. Simulate a **CeO2 powder diffraction image** — narrow Gaussian rings at
   each Bragg position via ImageD11 `compute_tth_eta`.
3. **Histogram** the data with ImageD11's native geometry and pyFAI's module‑gap mask.
   All angles in **degrees**.
4. Discover **all 32 pyFAI poni conversion solutions** via
   `find_all_poni_solutions()`.
5. Show the **4 canonical solutions** one per cell — each with a
   2‑panel figure (1D + 2D integrations) and all equivalent `repr(ai)` descriptions.
6. Plot every solution individually in a compact grid, then print the
   full `repr(ai)` for all 32.

### Geometry

| Parameter | Value |
|---|---|
| Detector | Eiger4M `(2167, 2070)`, 75 µm pixels |
| Beam centre | `(y=123, z=456)` px |
| Tilts | `tx=0.04, ty=0.03, tz=0.02` rad |
| Wavelength | 0.31 Å |
| Distance | ≈ 500 mm |
| Flips | `(−1, 0, 0, −1)` → orientation 4 |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import sys; sys.path.insert(0, "..")
import par_to_poni as pp
from ImageD11.transform import compute_tth_eta
from pyFAI import detector_factory
from pyFAI.integrator.azimuthal import AzimuthalIntegrator
from par_to_poni_full import find_all_poni_solutions

print("Imports OK")

## 1. Define the ImageD11 geometry

In [ ]:
det = detector_factory("Eiger4M")
mask = det.mask                  # int8: 0 = valid, 1 = gap
mask_bool = mask.astype(bool)
shape = det.shape                # (2167, 2070)

print(f"Eiger4M: shape={det.shape}, pixel={det.pixel1 * 1e6:.1f} µm")
print(f"  gaps: {mask.sum()} / {mask.size} px ({100 * mask.sum() / mask.size:.1f} %)")

par = {
    "distance": 0.5, "y_center": 123.0, "z_center": 456.0,
    "y_size": 75e-6, "z_size": 75e-6,
    "tilt_x": 0.04, "tilt_y": 0.03, "tilt_z": 0.02,
    "o11": -1, "o12": 0, "o21": 0, "o22": -1,
    "wavelength": 0.31e-10,
    "wedge": 0.0, "chi": 0.0, "omegasign": 1.0, "fit_tolerance": 0.05,
}

orient = pp.flip_to_orientation(par["o11"], par["o12"], par["o21"], par["o22"])
print(f"\nFlip ({par['o11']},{par['o12']},{par['o21']},{par['o22']}) "
      f"→ canonical orient {orient}")
print(f"Beam: ({par['y_center']}, {par['z_center']}) px  "
      f"dist={par['distance']*1000:.0f} mm  λ={par['wavelength']*1e10:.2f} Å")
print(f"Tilts: tx={par['tilt_x']:.3f} ty={par['tilt_y']:.3f} tz={par['tilt_z']:.3f} rad")

## 2. CeO2 ideal Bragg positions

In [ ]:
def ceo2_reflections(wavelength_m, max_tth=0.30):
    a = 5.411;  wl_A = wavelength_m * 1e10
    peaks = [];  seen_hkl = set()
    for n in range(3, 28):
        d = a / np.sqrt(n)
        sinth = wl_A / (2 * d)
        if sinth >= 1.0: continue
        tth = 2 * np.arcsin(sinth)
        if tth > max_tth: continue
        hkl_key = None
        for h in range(6):
            for k in range(6):
                for l in range(6):
                    if h*h + k*k + l*l != n: continue
                    if (h%2, k%2, l%2) not in ((0,0,0), (1,1,1)): continue
                    key = tuple(sorted((h,k,l)))
                    if key in seen_hkl or hkl_key: continue
                    hkl_key = key
        if hkl_key is None: continue
        seen_hkl.add(hkl_key)
        h,k,l = sorted(hkl_key)
        peaks.append((f"({h}{k}{l})", d, tth))
    return sorted(peaks, key=lambda x: x[2])

peaks = ceo2_reflections(par["wavelength"])
print(f"CeO2 (λ={par['wavelength']*1e10:.2f} Å):")
for hkl, d, tth in peaks:
    print(f"  {hkl:>6s}  d={d:.4f} Å  2θ={np.degrees(tth):.3f}°")
print(f"  ... {len(peaks)} peaks")

## 3. Build the CeO2 powder image

In [ ]:
slow_s, fast_s = shape
sc, fc = np.meshgrid(np.arange(slow_s, dtype=np.float64),
                     np.arange(fast_s, dtype=np.float64), indexing="ij")

pid = {k: v for k, v in par.items() if k not in ("fit_tolerance",)}
pid["wavelength"] = par["wavelength"] * 1e10
tth_deg_flat, eta_deg_flat = compute_tth_eta((sc.ravel(), fc.ravel()), **pid)
tth_deg = tth_deg_flat.reshape(shape)
eta_deg = eta_deg_flat.reshape(shape)
tth_rad = np.radians(tth_deg)

print(f"tth range: {tth_deg[mask_bool == False].min():.2f} – "
      f"{tth_deg[mask_bool == False].max():.2f}°")

ring_scales = {
    "(111)": 1.0, "(200)": 0.15, "(220)": 0.55, "(311)": 0.40,
    "(222)": 0.10, "(400)": 0.20, "(331)": 0.25, "(420)": 0.15,
}
sigma_tth = 0.0012;  rng = np.random.RandomState(42)

image_clean = np.zeros(shape, dtype=np.float64)
for hkl, d, tth_i in peaks:
    gauss = np.exp(-0.5 * ((tth_rad - tth_i) / sigma_tth) ** 2)
    image_clean += ring_scales.get(hkl, 0.3) * gauss
image_clean += np.maximum(0.02 * (1 + 0.1 * rng.randn(*shape)), 0)
image_noisy = rng.poisson(np.maximum(image_clean * 500, 0)).astype(np.float64)

image = np.where(mask_bool, np.nan, image_noisy)
image_for_pyfai = np.where(np.isnan(image), 0, image)

valid = ~np.isnan(image)
print(f"Valid pixels: {valid.sum():,} / {valid.size:,}")

## 4. Display the image

In [ ]:
vlo, vhi = np.nanpercentile(image, [2, 98])
fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(image, origin="lower", cmap="inferno", vmin=vlo, vmax=vhi)
ax.set_title(f"Synthetic CeO2 on Eiger4M\n"
             f"beam=({par['y_center']},{par['z_center']}) px  "
             f"dist={par['distance']*1000:.0f} mm")
ax.set_xlabel("Fast axis (px)");  ax.set_ylabel("Slow axis (px)")
plt.colorbar(ax.images[0], ax=ax, label="Photons", shrink=0.78)
plt.tight_layout();  plt.show()

## 5. ImageD11 histogram — normalised 1D and 2D 2θ/η  (degrees)

In [ ]:
# Normalised 1D radial average:  ΣI / Σcount  per 2θ bin
npt_rad, npt_azim = 1000, 360

tth_flat = tth_deg[valid];  eta_flat = eta_deg[valid]
img_flat  = image[valid]

edges_1d = np.linspace(tth_flat.min(), tth_flat.max(), npt_rad + 1)
center_1d = 0.5 * (edges_1d[:-1] + edges_1d[1:])
sum_I, _ = np.histogram(tth_flat, bins=edges_1d, weights=img_flat)
count,  _ = np.histogram(tth_flat, bins=edges_1d)
hist_norm = sum_I / np.maximum(count, 1)           # average intensity per pixel
hist_raw, _ = np.histogram(tth_flat, bins=edges_1d, weights=img_flat)

# 2D histogram
hist_2d, tth_e, eta_e = np.histogram2d(
    tth_flat, eta_flat, bins=[int(npt_rad/2), int(npt_azim/2)], weights=img_flat)

# Shareable helpers
def add_peak_lines(ax):
    for _, _, tth_i in peaks:
        ax.axvline(np.degrees(tth_i), color="cyan", ls="--", alpha=0.4, lw=0.6)

def _integ1d(ai, img, msk, npt=npt_rad, **kw):
    return ai.integrate1d(img, npt=npt, mask=msk, method="splitpixel",
                          unit="2th_deg", **kw)

def _integ2d(ai, img, msk, npt_rad=npt_rad, npt_azim=npt_azim):
    return ai.integrate2d(img, npt_rad=npt_rad, npt_azim=npt_azim,
                          mask=msk, method="splitpixel", unit="2th_deg")

# --- ImageD11 3-panel plot ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
ax.imshow(image, origin="lower", cmap="inferno", vmin=vlo, vmax=vhi)
ax.set_title("CeO2 image");  ax.set_xlabel("Fast (px)");  ax.set_ylabel("Slow (px)")

ax = axes[1]
ax.plot(center_1d, hist_norm)
add_peak_lines(ax)
ax.set_xlabel("2θ (°)");  ax.set_ylabel("⟨I⟩ per pixel")
ax.set_title("ImageD11 1D — normalised radial average")

ax = axes[2]
pc = ax.pcolormesh(tth_e, eta_e, hist_2d.T, cmap="inferno", shading="auto")
add_peak_lines(ax)
ax.set_xlabel("2θ (°)");  ax.set_ylabel("η (°)")
ax.set_title("ImageD11 2D — 2θ vs η")
plt.colorbar(pc, ax=ax)

plt.tight_layout();  plt.show()

## 6.  Solutions, helpers, and ImageD11 reference data
Everything needed for the per‑solution plots.

In [ ]:
solutions = pp.find_all_poni_solutions(par, detector_shape=shape)
print(f"Found {len(solutions)} poni solutions\n")

from collections import Counter
for (ot, ms), cnt in sorted(Counter(
    (s["orient_tried"], s["mirror_source"]) for s in solutions).items()):
    grp = [s for s in solutions if s["orient_tried"] == ot and s["mirror_source"] == ms]
    n_pos = sum(1 for s in grp if s["dist_positive"])
    n_chi = sum(1 for s in grp if s["chi_eta_exact"])
    print(f"  orient={ot}  mirror={ms}:  {cnt} reps, {n_pos} pos, {n_chi} chi-exact")

sorted_sols = sorted(solutions,
    key=lambda s: (s["orient_tried"], s["mirror_source"],
                   0 if s["dist_positive"] else 1))

def build_ai(pd, shp=shape):
    ai = AzimuthalIntegrator(
        dist=pd["dist"], poni1=pd["poni1"], poni2=pd["poni2"],
        rot1=pd["rot1"], rot2=pd["rot2"], rot3=pd["rot3"],
        pixel1=pd["pixel1"], pixel2=pd["pixel2"],
        wavelength=pd.get("wavelength"), orientation=pd["orientation"])
    ai.detector.shape = shp
    return ai

mask_int = mask.astype(np.int8)

# Identify canonical + default solutions
canonical = {}
for s in solutions:
    if s["chi_eta_exact"]:
        ot = s["orient_tried"]
        if ot not in canonical: canonical[ot] = s

default_poni = pp.par_to_poni(par, detector_shape=shape)
is_default = lambda s: (
    abs(s["poni"]["dist"] - default_poni["dist"]) < 1e-10 and
    abs(s["poni"]["rot1"] - default_poni["rot1"]) < 1e-10 and
    abs(s["poni"]["rot2"] - default_poni["rot2"]) < 1e-10 and
    abs(s["poni"]["rot3"] - default_poni["rot3"]) < 1e-10)

def equiv_indices(canon_sol):
    mo = canon_sol["poni"].get("_mirror_orient", canon_sol["poni"]["orientation"])
    dp = canon_sol["dist_positive"]
    return [i for i, s in enumerate(sorted_sols)
            if s["poni"].get("_mirror_orient", s["poni"]["orientation"]) == mo
            and s["dist_positive"] == dp]

mirror_info = {
    1: ("M1 (−I)", "χ = 270°−η"),
    2: ("M2",       "χ = η−90°"),
    3: ("M3 (+I)",  "χ = 90°−η"),
    4: ("M4",       "χ = η+90°"),
}

# --- Shared two-panel plot function ---
def plot_solution(sol, equiv_sols, title_extra=""):
    """2-panel figure: (L) 1D overlay + ImageD11 ref, (R) 2D integrate.
    Returns the figure."""
    ot = sol["orient_tried"];  p = sol["poni"]
    m_label, chi_formula = mirror_info.get(
        p.get("_mirror_orient", ot), mirror_info[ot])
    pos = "+" if sol["dist_positive"] else "−"
    dm = " ★ DEFAULT" if is_default(sol) else ""

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Left: ImageD11 reference on y2 + all equivalent pyFAI on y1
    ax1b = ax1.twinx()
    ax1b.plot(center_1d, hist_norm, "k-", alpha=0.35, lw=1.5)
    ax1b.set_ylabel("ImageD11  ⟨I⟩ per pixel", color="0.3");  ax1b.tick_params(colors="0.3")

    for es in equiv_sols:
        res = _integ1d(build_ai(es["poni"]), image_for_pyfai, mask_int)
        ax1.plot(res.radial, res.intensity, alpha=0.4, lw=0.6)
    add_peak_lines(ax1)
    ax1.set_xlabel("2θ (°)");  ax1.set_ylabel("pyFAI intensity")
    ax1.set_title(f"1D — ImageD11 (y2) + {len(equiv_sols)} pyFAI solutions (y1)")

    # Right: 2D integrate2d
    res2d = _integ2d(build_ai(p), image_for_pyfai, mask_int)
    ax2.pcolormesh(res2d.radial, res2d.azimuthal, res2d.intensity,
                   cmap="inferno", shading="auto", rasterized=True)
    add_peak_lines(ax2)
    ax2.set_xlabel("2θ (°)");  ax2.set_ylabel("χ (°)")
    ax2.set_title(f"2D — {m_label}, {chi_formula}")

    suptitle = (f"Canonical orient={ot}  {m_label}  dist{pos}{dm}"
                f"{'  ' + title_extra if title_extra else ''}")
    fig.suptitle(suptitle, fontsize=12, fontfamily="monospace")
    fig.tight_layout()
    return fig

def print_equiv_reprs(sol, equiv_sols):
    """Print all equivalent poni repr(ai) descriptions."""
    print(f"\n{len(equiv_sols)} equivalent poni descriptions:")
    print("=" * 75)
    for i, es in enumerate(equiv_sols):
        p = es["poni"];  dm = " ★ DEFAULT" if is_default(es) else ""
        print(f"\n  [#{i}]  orient={es['orient_tried']}  "
              f"chi_exact={es['chi_eta_exact']}{dm}")
        print(f"  {repr(build_ai(p))}")

print(f"\nDefault par_to_poni(): orientation {default_poni['orientation']}, "
      f"mirror {mirror_info[default_poni.get('_mirror_orient', default_poni['orientation'])][0]}, "
      f"dist={default_poni['dist']:.4f} (smallest rot magnitude among pos-dist)\n")

## 7. All‑32 1D overlay — 2θ identical

Every solution gives the same radial integration.
Colour = pyFAI orientation.

In [ ]:
colors = {1: "tab:blue", 2: "tab:orange", 3: "tab:green", 4: "tab:red"}
fig, ax = plt.subplots(figsize=(10, 5))
for s in sorted_sols:
    res = _integ1d(build_ai(s["poni"]), image_for_pyfai, mask_int)
    ax.plot(res.radial, res.intensity, color=colors[s["orient_tried"]],
            alpha=0.3, lw=0.6)
add_peak_lines(ax)
from matplotlib.lines import Line2D
ax.legend(handles=[Line2D([0],[0], color=colors[o], label=f"orient {o}")
                    for o in [1,2,3,4]])
ax.set(xlabel="2θ (°)", ylabel="Intensity",
       title="All 32 solutions — 2θ identical (curves overlap)")
plt.tight_layout();  plt.show()

## 8. Solution clustering

The four mirror matrices split into two **pairs** related by a 180° χ offset:

| Pair | Mirrors | χ | sin χ | cos χ |
|------|---------|---|-------|-------|
| Type A | M3 (+I), M1 (−I) | 90°−η  /  270°−η | ±cos η | ±sin η |
| Type B | M2, M4           | η−90°  /  η+90°   | ∓cos η | ±sin η |

M1 = −M3 and M4 = −M2 on the first two diagonals, so sin χ
and cos χ both flip sign → 180° χ shift (unresolvable from
powder data alone).

Solutions with the **same mirror** and **same distance sign**
produce identical integrations.  Negative distances invert the
intensity — an Euler‑angle artefact.

Below, each canonical (`chi_eta_exact`) solution gets its own cell
with a 2‑panel figure (1D overlay + 2D integrate) and all
equivalent `repr(ai)` descriptions.  ★ marks the default from `par_to_poni()`.
The solution closest to the default is shown **first**.

## 9a.  Canonical orient = 2,  M2,  χ = η−90°

In [ ]:
ot = 2
s = canonical[ot]
eq_idx = equiv_indices(s)
eq_sols = [sorted_sols[i] for i in eq_idx]
m_label, chi_formula = mirror_info[ot]

pos = "+" if s["dist_positive"] else "−"
dm = " ★ DEFAULT" if is_default(s) else ""
print(f"Mirror: {m_label},  {chi_formula},  dist{pos}{dm}")
print(f"  {len(eq_sols)} equivalent solutions: #{eq_idx}")

fig = plot_solution(s, eq_sols)
plt.show()

print_equiv_reprs(s, eq_sols)

## 9b.  Canonical orient = 1,  M1 (−I),  χ = 270°−η

In [ ]:
ot = 1
s = canonical[ot]
eq_idx = equiv_indices(s)
eq_sols = [sorted_sols[i] for i in eq_idx]
m_label, chi_formula = mirror_info[ot]

pos = "+" if s["dist_positive"] else "−"
dm = " ★ DEFAULT" if is_default(s) else ""
print(f"Mirror: {m_label},  {chi_formula},  dist{pos}{dm}")
print(f"  {len(eq_sols)} equivalent solutions: #{eq_idx}")

fig = plot_solution(s, eq_sols)
plt.show()

print_equiv_reprs(s, eq_sols)

## 9c.  Canonical orient = 3,  M3 (+I),  χ = 90°−η

In [ ]:
ot = 3
s = canonical[ot]
eq_idx = equiv_indices(s)
eq_sols = [sorted_sols[i] for i in eq_idx]
m_label, chi_formula = mirror_info[ot]

pos = "+" if s["dist_positive"] else "−"
dm = " ★ DEFAULT" if is_default(s) else ""
print(f"Mirror: {m_label},  {chi_formula},  dist{pos}{dm}")
print(f"  {len(eq_sols)} equivalent solutions: #{eq_idx}")

fig = plot_solution(s, eq_sols)
plt.show()

print_equiv_reprs(s, eq_sols)

## 9d.  Canonical orient = 4,  M4,  χ = η+90°

In [ ]:
ot = 4
s = canonical[ot]
eq_idx = equiv_indices(s)
eq_sols = [sorted_sols[i] for i in eq_idx]
m_label, chi_formula = mirror_info[ot]

pos = "+" if s["dist_positive"] else "−"
dm = " ★ DEFAULT" if is_default(s) else ""
print(f"Mirror: {m_label},  {chi_formula},  dist{pos}{dm}")
print(f"  {len(eq_sols)} equivalent solutions: #{eq_idx}")

fig = plot_solution(s, eq_sols)
plt.show()

print_equiv_reprs(s, eq_sols)

## 10. All 32 solutions — individual 2D integrate2d grid

In [ ]:
n_rows, n_cols = 4, 8
fig, axes = plt.subplots(n_rows, n_cols, figsize=(24, 13), dpi=80)
fig.subplots_adjust(wspace=0.35, hspace=0.55)

for idx, s in enumerate(sorted_sols):
    row, col = divmod(idx, n_cols)
    ax = axes[row][col]
    p = s["poni"]
    res = _integ2d(build_ai(p), image_for_pyfai, mask_int)

    ax.pcolormesh(res.radial, res.azimuthal, res.intensity,
                  cmap="inferno", shading="auto", rasterized=True)
    add_peak_lines(ax)

    pos = "+" if s["dist_positive"] else "−"
    chx = "χ≡η" if s["chi_eta_exact"] else ""
    dm  = " ★" if is_default(s) else ""
    ax.set_title(
        f"#{idx}{dm} o={s['orient_tried']} {s['mirror_source']} "
        f"dist{pos} {chx}\n"
        f"d={p['dist']:.4f} r={{{p['rot1']:.2f},{p['rot2']:.2f},{p['rot3']:.2f}}}",
        fontsize=6, fontfamily="monospace", pad=2)
    ax.set_xlabel("2θ (°)", fontsize=6);  ax.set_ylabel("χ (°)", fontsize=6)
    ax.tick_params(labelsize=5)

fig.suptitle("pyFAI integrate2d — all 32 (★ = default)", fontsize=13, y=1.01)
fig.tight_layout();  plt.show()

## 11. Print `repr(ai)` for all 32 solutions

In [ ]:
print("=" * 82)
print(f"{'repr(AzimuthalIntegrator) for all {len(sorted_sols)} solutions':^82}")
print("=" * 82)

for i, s in enumerate(sorted_sols):
    p = s["poni"];  dm = " ★ DEFAULT" if is_default(s) else ""
    print(f"\n--- Solution {i:2d}  "
          f"orient={s['orient_tried']}  mirror={s['mirror_source']}  "
          f"chi_exact={s['chi_eta_exact']}  pos_dist={s['dist_positive']}{dm}")
    print(repr(build_ai(p)))

## 12. Summary

- **All 32 solutions** produce **identical 2θ** — the 1D curves in §7
  overlap to numerical precision.

- **Azimuth (χ)** differs by mirror: the 4 mirrors form two **χ‑pairs**
  (M3/M1 and M2/M4) differing by 180° within each pair.
  See the canonical comparisons in §9a–9d.

- **Negative‑distance** solutions flip cos rot₁ · cos rot₂ without
  changing the rotation matrix itself (Euler‑angle equivalence).

- The **default** (★) returned by `par_to_poni()` picks the smallest
  rotation magnitude among positive‑distance candidates.  For this
  geometry that is `orient = 1, M2` (shown first in §9a).

- **Module gaps** from `pyFAI.detector.Eiger4M.mask` are handled
  everywhere.